In [1]:
# Import from main and experiments library
import os
from exp_library import *
os.chdir("../")
from library import *

# filter the warnings for clarity
import warnings
warnings.filterwarnings("ignore")

## Bankruptcy Prediction - Ensemble Model 💸📜📈

### Introduction

In this notebook, we include the experiments for the next-year bankruptcy prediction task using numerical and textual predictors from the following paper:

```Henri Arno, Klaas Mulier, Joke Baeck, and Thomas Demeester. 2023. From Numbers to Words: Multi-Modal Bankruptcy Prediction Using the ECL Dataset. In Proceedings of the Sixth Workshop on Financial Technology and Natural Language Processing (FinNLP), Bali, Indonesia. Association for Computational Linguistics.```


For an explanation of the task, see the notebook *Numerical Models.ipynb*. The figure below shows the task intuitively.

<div>
<img src="../images/task.png" width="800" align="left"/>
</div>

### Read in the data

In [2]:
# Read in the dataset.
dataset = pd.read_csv('ECL.csv', index_col=0)

### Add numerical and textual predictors

#### Option 1: Extract CompuStat data via the WRDS Python API

In [ ]:
# Login using WRDS credentials.
username = ''
db = wrds.Connection(wrds_username=username)

# List the required variables
variables = 'act, lct, ap, sale, che, lt, mib, ch, ebit, dp, dlc, dltt, invch, invt, ni, oiadp, re, seq, wcap, at'

# Add the CompuStat variables to the MultiModalFinance dataset.
dataset = get_CompuStat_WRDS(variables, dataset, db)

#### Option 2: Extract CompuStat data via a local copy

In [3]:
# Path to the CompuStat data.
path = './data/CompuStat/data.csv'

# Add the CompuStat variables to the MultiModalFinance dataset.
dataset = get_CompuStat_local(path, dataset, update=False)

Dropped 115373 rows from CompuStat based on screening variables
0 records in the dataset do not have an accompanying CompuStat record.


#### Compute numerical features

In [4]:
# Compute features and store the column indices.
dataset, predictors = compute_features(dataset)

# Specify path to the corpus of cleaned documents
# See the "Textual Models.ipynb" notebook for more details
clean_corpus = './data/clean_corpus'

### Create uni-modal models (base classifiers)

In [5]:
# Store the records that can be labelled and are qualified for inclusion in the LoPucki BRD.
prediction_subset = dataset.loc[(dataset['can_label'] == True) & (dataset['qualified'] == 'Yes')].reset_index(drop=True)
prediction_subset['filename'] = prediction_subset['filename'].str.replace('.json', '.txt')

# Split the data in a train, test and validation set
prediction_subset['fyear'] = pd.to_datetime(prediction_subset['filing_date']).dt.year
# split in train and test
train_full = prediction_subset.loc[prediction_subset['bankruptcy_prediction_split'] == 'train']
val = train_full.loc[train_full['fyear'] > 2011]
train_small = train_full.drop(val.index)
test = prediction_subset.loc[prediction_subset['bankruptcy_prediction_split'] == 'test']

#### Numerical: XGBoost

In [6]:
# Store predictors and labels
small_X = train_small[predictors]
small_y = train_small['label']

val_X = val[predictors]
val_y = val['label']

full_X = train_full[predictors]
full_y = train_full['label']

test_X = test[predictors]
test_y = test['label']

# resample training data
ros = RandomOverSampler(random_state=0, sampling_strategy=1)
small_X, small_y = ros.fit_resample(small_X, small_y)
full_X, full_y = ros.fit_resample(full_X, full_y)

In [7]:
# create the pipelines
XGB_small = Pipeline([ ('scaler', StandardScaler()), 
                      ('clf', xgb.XGBClassifier(objective='binary:logistic', subsample=0.5, eta=0.1, 
                                                max_depth = 1, n_estimators = 1000))])

XGB_full = Pipeline([ ('scaler', StandardScaler()), 
                      ('clf', xgb.XGBClassifier(objective='binary:logistic', subsample=0.5, eta=0.1, 
                                                max_depth = 1, n_estimators = 1000))])

# train model
XGB_small.fit(X=small_X, y=small_y)
XGB_full.fit(X=full_X, y=full_y)

# evaluate the model
val_preds = XGB_small.predict_proba(val_X)[:, 1]
test_preds = XGB_full.predict_proba(test_X)[:, 1]
val["preds_XGB"] = val_preds
test["preds_XGB"] = test_preds

#### Textual: TF-IDF

In [8]:
# split predictors and labels
small_X = clean_corpus + train_small['filename']
small_y = train_small['label']

val_X = clean_corpus + val['filename']
val_y = val['label']

full_X = clean_corpus + train_full['filename']
full_y = train_full['label']

test_X = clean_corpus + test['filename']
test_y = test['label']

In [9]:
# create the pipeline
TF_IDF_small = Pipeline([
    ('vect', TfidfVectorizer(input='filename', lowercase=True, 
                                 strip_accents='ascii', stop_words='english', min_df=2, ngram_range = (1,2))),
    ('clf', LogisticRegression(penalty = 'l1', C = 1, class_weight = 'balanced', 
                                   solver='liblinear'))])

TF_IDF_full = Pipeline([
    ('vect', TfidfVectorizer(input='filename', lowercase=True, 
                                 strip_accents='ascii', stop_words='english', min_df=2, ngram_range = (1,2))),
    ('clf', LogisticRegression(penalty = 'l1', C = 1, class_weight = 'balanced', 
                                   solver='liblinear'))])

# train model
TF_IDF_small.fit(X=small_X, y= small_y)
TF_IDF_full.fit(X=full_X, y= full_y)

# evaluate the model
val_preds = TF_IDF_small.predict_proba(val_X)[:, 1]
test_preds = TF_IDF_full.predict_proba(test_X)[:, 1]
val["preds_TFIDF"] = val_preds
test["preds_TFIDF"] = test_preds

### Train and evaluate meta-classifier

In [12]:
# split predictors and labels
train_X = val[["preds_XGB", "preds_TFIDF"]]
train_y = val['label']

test_X = test[["preds_XGB", "preds_TFIDF"]]
test_y = test['label']

In [13]:
# create the pipelines
stacking_clf = Pipeline([('scaler', StandardScaler()),
                         ('clf', LogisticRegression(penalty='l2', C =1 , class_weight='balanced'))])

# train
stacking_clf.fit(X=train_X, y = train_y)

# evaluate the model
preds = stacking_clf.predict_proba(test_X)[:, 1]
evaluate(labels=test_y, predictions=preds)

-- RESULTS --
AUC: 0.9479
AP: 0.2639
recall@100: 0.2869
CAP: 0.8958
